In [ ]:
# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then work from this notebook's own folder, which is
# what the relative paths below assume.
import os
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
_here = _root / "refinement-per-claim" / "human-validation-initial"
if not _here.is_dir():
    raise RuntimeError(
        "Could not locate " + str(_here) + ". Run this notebook from inside the "
        "cloned repository."
    )
os.chdir(_here)

In [ ]:
import pandas as pd
import numpy as np

# Randomize 'cn_1' and 'cn_2' columns along with their scores
def randomize_cn_and_scores(df):
    # Updated score column map based on the inspection
    score_cols_map = {
        'cn_1': 'avg_score_cn_1',
        'cn_2': 'avg_score_cn_2'
    }

    df_copy = df.copy() # Make a copy to avoid modifying the original DataFrame directly
    for index, row in df_copy.iterrows():
        # Randomly decide whether to swap (50% chance)
        if np.random.rand() < 0.5:
            # Swap 'cn_1' and 'cn_2'
            df_copy.at[index, 'cn_1'], df_copy.at[index, 'cn_2'] = \
                df_copy.at[index, 'cn_2'], df_copy.at[index, 'cn_1']

            # Swap corresponding scores if they exist
            if score_cols_map['cn_1'] in df_copy.columns and score_cols_map['cn_2'] in df_copy.columns:
                df_copy.at[index, score_cols_map['cn_1']], df_copy.at[index, score_cols_map['cn_2']] = \
                    df_copy.at[index, score_cols_map['cn_2']], df_copy.at[index, score_cols_map['cn_1']]
    return df_copy

# Keep records sorted by 'claim' but shuffle within each 'claim'
def shuffle_within_group(df, group_col):
    return df.groupby(group_col).apply(lambda x: x.sample(frac=1, random_state=42)).reset_index(drop=True)

# Process KPI1
df_kpi1 = pd.read_excel('comparisons_from_refinement.xlsx', sheet_name='KPI1')
df_kpi1_randomized = randomize_cn_and_scores(df_kpi1)
df_kpi1_final = shuffle_within_group(df_kpi1_randomized, 'claim')

# Process KPI2
df_kpi2 = pd.read_excel('comparisons_from_refinement.xlsx', sheet_name='KPI2')
df_kpi2_randomized = randomize_cn_and_scores(df_kpi2)
df_kpi2_final = shuffle_within_group(df_kpi2_randomized, 'claim')

# Process KPI3
df_kpi3 = pd.read_excel('comparisons_from_refinement.xlsx', sheet_name='KPI3')
df_kpi3_randomized = randomize_cn_and_scores(df_kpi3)
df_kpi3_final = shuffle_within_group(df_kpi3_randomized, 'claim')

In [ ]:
df_kpi1_final.to_excel('kpi1_randomized.xlsx', index=False)
df_kpi2_final.to_excel('kpi2_randomized.xlsx', index=False)
df_kpi3_final.to_excel('kpi3_randomized.xlsx', index=False)